In [1]:
#Reading the Data files
import pandas as pd
import numpy as np
import statsmodels.api as sm
import seaborn as sns
import matplotlib.pyplot as plt

analysis = pd.read_csv('/Users/chintanjikkar/Desktop/PACE/Fall 25/Predictive Analytics/Assignments/PAC/Data/analysis_data.csv')
scoring = pd.read_csv('/Users/chintanjikkar/Desktop/PACE/Fall 25/Predictive Analytics/Assignments/PAC/Data/scoring_data.csv')

In [2]:
#Importing Libraries and functions to be used 
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [3]:
#Splitting the analysis data
train = analysis.sample(frac=0.7, random_state=1031)
test = analysis.drop(labels = train.index)

In [4]:
y_train = train['monthly_spend']
X_train = train.drop('monthly_spend', axis=1)

In [5]:
#Rectifying the NAs
# add missingness indicators (helps models learn if "missing" is informative)
X_train['online_shopping_freq_was_na']  = X_train['online_shopping_freq'].isna().astype(int)
X_train['utility_payment_count_was_na'] = X_train['utility_payment_count'].isna().astype(int)
X_train['education_level_was_na']       = X_train['education_level'].isna().astype(int)

# numeric counts -> median inputes of the rest of data
X_train['online_shopping_freq']  = X_train['online_shopping_freq'].fillna(X_train['online_shopping_freq'].median())
X_train['utility_payment_count'] = X_train['utility_payment_count'].fillna(X_train['utility_payment_count'].median())

# categorical -> mode inputes of the rest of data
edu_mode = X_train['education_level'].mode(dropna=True)
edu_fill = edu_mode.iloc[0] if len(edu_mode) else 'Unknown'
X_train['education_level'] = X_train['education_level'].fillna(edu_fill)

In [6]:
#Segregating the categorical stuff
categorical_features = [
    'gender', 'marital_status', 'education_level',
    'region', 'employment_status', 'card_type'
]

In [7]:
#Creating dummies
X_train_dum = pd.get_dummies(
    X_train,
    columns=[c for c in categorical_features if c in X_train.columns],
    drop_first=True,
    prefix_sep='_'
)

In [8]:
#Converting bool to int for future use(justincase)
bool_cols = X_train_dum.select_dtypes(include=['bool']).columns
X_train_dum[bool_cols] = X_train_dum[bool_cols].astype(int)

X_ols = sm.add_constant(X_train_dum.drop(columns=['customer_id']))